# 🎙️ Arabic Meeting Summarizer — Local Hybrid Edition
### يشتغل 100% على جهازك — VS Code — بدون Colab
### ASR: Faster-Whisper · Diarization: Simple Clustering · Summary: **Fanar (فهم عربي) + Qwen3 (تقرير هيكلي)**

---

## 🗺️ Pipeline

```
YouTube URL  أو  ملف فيديو/صوت محلي
              ↓
[Cell 1] ⚙️ الإعدادات ← عدّل هنا فقط
              ↓
[Cell 2] مساعدات RAM / GPU
              ↓
[Cell 3] تحميل الصوت من يوتيوب أو ملف محلي → WAV
              ↓
[Cell 4] معالجة الصوت: 16 kHz · mono · normalize
              ↓
[Cell 5] ASR — Faster-Whisper → transcript + Diarization
              ↓
[Cell 6] Hybrid Summary:
         • Fanar (محلي عبر Ollama) — يفهم النص العربي ويستخرج المعلومات
         • Qwen3 (محلي عبر Ollama) — يبني التقرير الهيكلي الكامل
              ↓
[Cell 7] عرض التقرير مرئياً (HTML يفتح في المتصفح)
              ↓
[Cell 8] تحرير الذاكرة
              ↓
[Cell 9-13] 📊 تقييم: ROUGE + BERTScore + LLM Judge (Qwen3)
```

---

## ✅ المتطلبات قبل التشغيل

### 1. تثبيت المكتبات (مرة واحدة في Terminal)
```bash
pip install faster-whisper yt-dlp imageio-ffmpeg soundfile torchaudio openai rouge-score bert-score psutil numpy ipykernel notebook
```
> `ipykernel` ضروري حتى يظهر بيئتك (environment) كخيار Kernel داخل VS Code.
> اختاري هذا الـ Kernel من الزاوية العلوية اليمنى لملف الـ notebook قبل التشغيل.

### 2. تأكد أن Ollama شغّال وعنده الموديلين
```bash
ollama serve
ollama list   # تأكد من وجود Fanar و Qwen3
```

### 3. أسماء الموديلات في Ollama (من `ollama list`)
ضعها في Cell 1 في `FANAR_MODEL_ID` و `QWEN_MODEL_ID`

> ⚠️ **شغّل الخلايا واحدة تلو الأخرى**


In [2]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"



In [4]:
# ════════════════════════════════════════════════════
# CELL 1 — ⚙️ الإعدادات — عدّل هنا فقط
# ════════════════════════════════════════════════════
import os
from datetime import datetime

# ── المدخل: يوتيوب أو ملف محلي ─────────────────────
YOUTUBE_URL = "https://www.youtube.com/watch?v=oueP7_CBIoo&t=1s"
LOCAL_FILE  = ""   # مثال: r"C:\meetings\meeting.mp4"
               #   اتركه فارغاً لو هتستخدم يوتيوب
               #   يقبل أي صيغة صوت/فيديو (mp4, mp3, m4a, wav, ...)

# ── إعدادات Whisper ──────────────────────────────────
USE_FAST_MODE     = False    # True = أسرع | False = أدق
NUM_SPEAKERS_HINT = None   # None = تلقائي | أو رقم مثل 2 أو 3
WHISPER_MODEL_SIZE = "medium"  # جرّبي: tiny/base/small/medium/large-v3-turbo
                               # كل ما زاد الحجم زادت الدقة وزاد وقت التحميل/التشغيل

# ── Ollama — Fanar (فهم عربي) ───────────────────────
# شغّل `ollama list` وانسخ اسم Fanar بالظبط
FANAR_MODEL_ID  = "fanar-8k"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# ── Ollama — Qwen3 (تقرير هيكلي) ───────────────────
# شغّل `ollama list` وانسخ اسم Qwen3 بالظبط
QWEN_MODEL_ID = "qwen3:latest"   # مثال: qwen3:8b أو qwen3:14b

# ── ملخص مرجعي للتقييم (اختياري) ───────────────────
REFERENCE_SUMMARY = "".strip()

# ── مسار الإخراج ────────────────────────────────────
OUTPUT_DIR = "./meeting_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
SESSION    = datetime.now().strftime('%Y%m%d_%H%M%S')

INPUT_MODE = "local" if LOCAL_FILE.strip() else "youtube"

print(f"{'='*55}")
print(f"  🎙️  Arabic Meeting Summarizer — Local Hybrid")
print(f"{'='*55}")
print(f"  Session:    {SESSION}")
print(f"  Input:      {INPUT_MODE.upper()}")
print(f"  Whisper:    {WHISPER_MODEL_SIZE}")
print(f"  Fanar:      {FANAR_MODEL_ID}")
print(f"  Qwen3:      {QWEN_MODEL_ID}")
print(f"  Output dir: {os.path.abspath(OUTPUT_DIR)}")
print(f"{'='*55}")


  🎙️  Arabic Meeting Summarizer — Local Hybrid
  Session:    20260724_201600
  Input:      YOUTUBE
  Whisper:    medium
  Fanar:      fanar-8k
  Qwen3:      qwen3:latest
  Output dir: c:\Users\Admin\Desktop\GP\meeting_output


In [5]:
# ════════════════════════════════════════════════════
# CELL 2 — مساعدات RAM / GPU
# ════════════════════════════════════════════════════
import psutil, torch, gc

def show_ram():
    ram    = psutil.virtual_memory()
    filled = int(ram.percent / 5)
    bar    = '█' * filled + '░' * (20 - filled)
    print(f'RAM  [{bar}] {ram.used/1e9:.1f}/{ram.total/1e9:.1f} GB  ({ram.percent:.0f}%)')
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'GPU  {alloc:.1f}/{total:.1f} GB  |  CUDA {torch.version.cuda}')
    else:
        print('GPU  not available — CPU mode')

def free_memory(*vars_to_delete):
    for v in vars_to_delete:
        try: del v
        except: pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print('🧹 Memory freed')

show_ram()
print("✅ مساعدات الذاكرة جاهزة")


RAM  [██████████░░░░░░░░░░] 17.8/34.0 GB  (52%)
GPU  not available — CPU mode
✅ مساعدات الذاكرة جاهزة


In [ ]:
# ════════════════════════════════════════════════════
# CELL 3 — تحميل الصوت (يوتيوب أو ملف محلي)
# ════════════════════════════════════════════════════
import os, gc, subprocess
from pathlib import Path
import yt_dlp, imageio_ffmpeg

free_memory()
FFMPEG_BIN = imageio_ffmpeg.get_ffmpeg_exe()
AUDIO_PATH = None

if INPUT_MODE == "local":
    if not os.path.exists(LOCAL_FILE):
        raise FileNotFoundError(f"❌ الملف غير موجود: {LOCAL_FILE}")
    print(f"📂 ملف محلي: {LOCAL_FILE}")

    # تحويل أي صيغة صوت/فيديو إلى WAV عبر ffmpeg
    # (soundfile في Cell 4 لا يقرأ إلا WAV/FLAC/OGG، فلازم نحوّل mp4/mp3/... أولاً)
    local_wav = f"{OUTPUT_DIR}/{SESSION}_local.wav"
    print("🔄 تحويل الملف المحلي إلى WAV ...")
    cmd = [FFMPEG_BIN, "-y", "-i", LOCAL_FILE, "-ac", "1", "-ar", "16000", local_wav]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0 or not os.path.exists(local_wav):
        raise RuntimeError(f"❌ فشل تحويل الملف بواسطة ffmpeg:\n{result.stderr[-1500:]}")
    AUDIO_PATH = local_wav
    print("✅ تم التحويل إلى WAV")

else:
    print(f"⬇️  جاري تحميل: {YOUTUBE_URL}")
    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": f"{OUTPUT_DIR}/{SESSION}_%(id)s.%(ext)s",
        "ffmpeg_location": FFMPEG_BIN,
        "postprocessors": [{"key": "FFmpegExtractAudio",
                            "preferredcodec": "wav", "preferredquality": "192"}],
        "quiet": False,
        "no_warnings": True,
        "socket_timeout": 30,
        "restrictfilenames": True,
        "extractor_args": {"youtube": {"player_client": ["android", "web"]}},
    }
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(YOUTUBE_URL, download=True)
        wavs = sorted(Path(OUTPUT_DIR).glob(f"{SESSION}_*.wav"))
        if not wavs:
            raise FileNotFoundError("لم يُوجد ملف WAV بعد التحميل")
        AUDIO_PATH = str(wavs[-1])
        print(f"✅ تم التحميل: {info.get('title','Unknown')}")
    except Exception as e:
        print(f"❌ فشل التحميل: {e}")
        print("   💡 ضع مسار ملف محلي في LOCAL_FILE بـ Cell 1")
        raise

size_mb = os.path.getsize(AUDIO_PATH) / 1e6
print(f"📁 الملف الصوتي: {AUDIO_PATH} ({size_mb:.1f} MB)")


🧹 Memory freed
⬇️  جاري تحميل: https://www.youtube.com/watch?v=oueP7_CBIoo&t=1s
[youtube] Extracting URL: https://www.youtube.com/watch?v=oueP7_CBIoo&t=1s
[youtube] oueP7_CBIoo: Downloading webpage
[youtube] oueP7_CBIoo: Downloading android player API JSON
[youtube] oueP7_CBIoo: Downloading web client config
[youtube] oueP7_CBIoo: Downloading web player API JSON
[info] oueP7_CBIoo: Downloading 1 format(s): 18
[download] Destination: meeting_output\20260724_201600_oueP7_CBIoo.mp4
[download]  31.0% of   13.72MiB at   17.11KiB/s ETA 09:26   

In [6]:
# ════════════════════════════════════════════════════
# CELL 4 — معالجة الصوت: 16 kHz · mono · normalize
# ════════════════════════════════════════════════════
import gc, numpy as np, soundfile as sf, torch
import torchaudio.transforms as T

gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

audio_array, sample_rate = sf.read(AUDIO_PATH, dtype='float32')
print(f"   مُحمَّل: {sample_rate} Hz | shape: {audio_array.shape}")

# Mono
if audio_array.ndim > 1:
    audio_array = np.mean(audio_array, axis=1)
    print("   ✅ تحويل إلى Mono")

# Resample → 16 kHz
TARGET_SR = 16000
if sample_rate != TARGET_SR:
    print(f"   Resampling: {sample_rate} → {TARGET_SR} Hz ...")
    audio_tensor = torch.from_numpy(audio_array.copy()).float()
    audio_array  = T.Resample(int(sample_rate), TARGET_SR)(audio_tensor).numpy()
    sample_rate  = TARGET_SR

# Normalize
peak = np.abs(audio_array).max()
if peak > 0:
    audio_array = audio_array / (peak + 1e-8)

DURATION_MIN = len(audio_array) / TARGET_SR / 60
PROCESSED_WAV = f"{OUTPUT_DIR}/{SESSION}_processed.wav"
sf.write(PROCESSED_WAV, audio_array, TARGET_SR, subtype='PCM_16')

print(f"✅ الصوت جاهز: {DURATION_MIN:.1f} دقيقة | {TARGET_SR} Hz | Mono")
print(f"   محفوظ: {PROCESSED_WAV}")
show_ram()


   مُحمَّل: 44100 Hz | shape: (322482176, 2)
   ✅ تحويل إلى Mono
   Resampling: 44100 → 16000 Hz ...
✅ الصوت جاهز: 121.9 دقيقة | 16000 Hz | Mono
   محفوظ: ./meeting_output/20260718_140022_processed.wav
RAM  [██████████████░░░░░░] 24.0/34.0 GB  (71%)
GPU  not available — CPU mode


In [7]:
# ════════════════════════════════════════════════════
# CELL 5 — ASR: Faster-Whisper (عربي + إنجليزي) + Diarization
# ════════════════════════════════════════════════════
import json, gc, sys, re
import torch
from faster_whisper import WhisperModel

sys.setrecursionlimit(2000)
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE   = "float16" if DEVICE == "cuda" else "int8"
BEAM_SIZE = 3 if USE_FAST_MODE else 5
print(f"🖥️  Device: {DEVICE} | Compute: {COMPUTE} | Beam: {BEAM_SIZE}")

print(f"⏳ تحميل Faster-Whisper ({WHISPER_MODEL_SIZE}) ...")
asr_model = WhisperModel(WHISPER_MODEL_SIZE, device=DEVICE, compute_type=COMPUTE)
print("✅ النموذج جاهز")

print("⏳ تحويل الكلام لنص — كشف اللغة تلقائياً ...")
segments_iter, info = asr_model.transcribe(
    PROCESSED_WAV,
    language="ar",              # حددي العربي صراحة بدل الكشف التلقائي لو أغلب الفيديو عربي
    task="transcribe",
    beam_size=BEAM_SIZE,
    vad_filter=True,
    vad_parameters={"min_silence_duration_ms": 500},
    condition_on_previous_text=False,   # ⬅️ بيمنع تراكم/تكرار الأخطاء بين المقاطع
    compression_ratio_threshold=2.4,    # ⬅️ بيرفض المقاطع اللي شكلها تكراري
    no_speech_threshold=0.6,
)
print(f"   اللغة الرئيسية: {info.language} (prob={info.language_probability:.2f})")
segments = list(segments_iter)
print(f"   ✅ {len(segments)} مقطع")

def detect_segment_lang(text):
    ar = len([c for c in text if '\u0600' <= c <= '\u06FF'])
    total = len([c for c in text if c.isalpha()])
    if total == 0: return 'ar'
    return 'ar' if ar / total >= 0.4 else 'en'

lang_stats = {'ar': 0, 'en': 0}

def simple_diarize(segs, pause_threshold=1.5, max_speakers=6):
    speaker_id, prev_end, labelled = 0, 0.0, []
    for seg in segs:
        if seg.start - prev_end > pause_threshold:
            speaker_id = (speaker_id + 1) % max_speakers
        lang = detect_segment_lang(seg.text)
        lang_stats[lang] = lang_stats.get(lang, 0) + 1
        labelled.append({
            "speaker": f"Speaker_{speaker_id + 1}",
            "start":   round(seg.start, 2),
            "end":     round(seg.end, 2),
            "text":    seg.text.strip(),
            "lang":    lang,
        })
        prev_end = seg.end
    return labelled

max_spk = NUM_SPEAKERS_HINT if NUM_SPEAKERS_HINT else 6
diarized_segments = simple_diarize(segments, max_speakers=max_spk)

transcript_lines = [
    f"[{s['speaker']}][{s['lang'].upper()}] ({s['start']}s-{s['end']}s): {s['text']}"
    for s in diarized_segments
]
full_transcript = "\n".join(transcript_lines)

print(f"\n📜 معاينة أول 5 مقاطع:")
for line in transcript_lines[:5]: print(f"  {line}")
print(f"\n🌐 إحصاء اللغات: عربي={lang_stats.get('ar',0)} | إنجليزي={lang_stats.get('en',0)}")

TRANSCRIPT_TXT  = f"{OUTPUT_DIR}/{SESSION}_transcript.txt"
TRANSCRIPT_JSON = f"{OUTPUT_DIR}/{SESSION}_transcript.json"
with open(TRANSCRIPT_TXT,  "w", encoding="utf-8") as f: f.write(full_transcript)
with open(TRANSCRIPT_JSON, "w", encoding="utf-8") as f:
    json.dump(diarized_segments, f, ensure_ascii=False, indent=2)

print(f"\n💾 Transcript: {TRANSCRIPT_TXT}")
print(f"✅ {len(diarized_segments)} مقطع | {len(full_transcript)} حرف")
free_memory(asr_model)
show_ram()


🖥️  Device: cpu | Compute: int8 | Beam: 5
⏳ تحميل Faster-Whisper (medium) ...
✅ النموذج جاهز
⏳ تحويل الكلام لنص — كشف اللغة تلقائياً ...
   اللغة الرئيسية: ar (prob=1.00)
   ✅ 2357 مقطع

📜 معاينة أول 5 مقاطع:
  [Speaker_2][AR] (2.06s-8.06s): في 2017 أو أوائل 2018 من بعد كده المسجل من بعد كده بينشر هذا
  [Speaker_2][AR] (8.06s-15.06s): فإحنا بصدد إن شاء الله إن إحنا هنبدأ بإذن الله إن إحنا نرفع المجلة دولي يعني خلال الشجور الجاي
  [Speaker_2][AR] (15.06s-21.06s): عشان ما تبقوش محتاجين إن إنتوا تروحوا تدوروا على مجلات برا تنشروا فيها فلوس كتير
  [Speaker_2][AR] (21.06s-27.61s): المجلة بتاعتنا أو يعني إيه بحثة أصلا إنت بعد ما بدخلص الرسالة
  [Speaker_2][AR] (27.61s-33.7s): بالاعتداء أن أنت تأخذ نقطة أو نقطتين يعني أكثر نقطة جلوظة أو واضحين أوي

🌐 إحصاء اللغات: عربي=2350 | إنجليزي=7

💾 Transcript: ./meeting_output/20260718_140022_transcript.txt
✅ 2357 مقطع | 162673 حرف
🧹 Memory freed
RAM  [████████████████░░░░] 27.8/34.0 GB  (82%)
GPU  not available — CPU mode


In [8]:
# ════════════════════════════════════════════════════
# CELL 6 — Hybrid Summary: Fanar (فهم عربي) + Qwen3 (تقرير)
# كلاهما محلي عبر Ollama — بدون إنترنت 🖥️
# ════════════════════════════════════════════════════
import gc, torch, time
from openai import OpenAI

gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

if 'full_transcript' not in globals() or not full_transcript.strip():
    raise RuntimeError("❌ full_transcript فارغ — شغّل Cell 5 أولاً")

# ── A) إعداد Fanar (محلي عبر Ollama) ─────────────────
fanar_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

# ── B) إعداد Qwen3 (محلي عبر Ollama) ─────────────────
qwen_client  = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

# ── اختبار الاتصال بـ Ollama ─────────────────────────
try:
    models_available = [m.id for m in fanar_client.models.list().data]
    print(f"✅ Ollama متصل | النماذج المتاحة: {models_available}")
    if FANAR_MODEL_ID not in models_available:
        print(f"⚠️  {FANAR_MODEL_ID} غير موجود — شغّل: ollama pull {FANAR_MODEL_ID}")
    if QWEN_MODEL_ID not in models_available:
        print(f"⚠️  {QWEN_MODEL_ID} غير موجود — شغّل: ollama pull {QWEN_MODEL_ID}")
except Exception as e:
    raise ConnectionError(f"❌ Ollama غير متصل: {e}\n   شغّل: ollama serve")

print(f"✅ Fanar جاهز  | {FANAR_MODEL_ID}")
print(f"✅ Qwen3 جاهز  | {QWEN_MODEL_ID}")


# ════════════════════════════════════════════════════
# الخطوة 1: Fanar يفهم النص العربي ويستخرج المعلومات
# ════════════════════════════════════════════════════
def fanar_understand(full_text, max_chars=10000):
    """Fanar يقرأ النص ويستخرج المعلومات الجوهرية — متخصص عربي."""
    messages = [
        {"role": "system", "content": (
            "أنت مساعد متخصص في تحليل اجتماعات العمل باللغة العربية الفصيحة. "
            "الاجتماع قد يحتوي على جمل بالعربية والإنجليزية أو مزيج منهما. "
            "حلّل النص وافهم المحتوى بكلتا اللغتين واستخرج المعلومات بدقة عالية."
        )},
        {"role": "user", "content": (
            f"حلّل نص الاجتماع التالي واستخرج:\n"
            f"1. المواضيع الرئيسية التي نوقشت\n"
            f"2. القرارات المتخذة مع ذكر المسؤول عن كل قرار\n"
            f"3. المهام الموكلة لكل شخص والمواعيد النهائية\n"
            f"4. النقاط الخلافية أو غير المحسومة\n"
            f"5. أسماء المشاركين والأدوار إن وُجدت\n"
            f"6. أي مصطلحات إنجليزية وردت واذكرها كما هي\n\n"
            f"نص الاجتماع:\n{full_text[:max_chars]}"
        )},
    ]
    t0 = time.time()
    resp = fanar_client.chat.completions.create(
    model=FANAR_MODEL_ID,
    messages=messages,
    max_tokens=1200,
    temperature=0.3,
    extra_body={"options": {"num_ctx": 8192}}   # ⬅️ زوّدي الـ context window
    )


# ════════════════════════════════════════════════════
# الخطوة 2: Qwen3 يبني التقرير الهيكلي الكامل
# ════════════════════════════════════════════════════
def qwen_report(fanar_analysis, full_text, max_chars=12000):
    """Qwen3 يبني التقرير الهيكلي بناءً على تحليل Fanar — نفس الـ Prompt الأصلي."""
    fanar_section = f"**تحليل Fanar العربي للاجتماع:**\n{fanar_analysis}\n\n" if fanar_analysis else ""
    prompt = f"""أنت محلل اجتماعات محترف متخصص باللغة العربية. الاجتماع قد يحتوي على عربية وإنجليزية.

{fanar_section}**النص الكامل للاجتماع:**
{full_text[:max_chars]}

أنشئ تقريراً شاملاً واحترافياً بالتنسيق التالي بالضبط:

## 1. جدول القرارات الرئيسية
| # | القرار | التفاصيل | المسؤول | الموعد |
|---|--------|----------|---------|--------|

## 2. جدول المهام والمسؤوليات
| # | المهمة | المسؤول | الموعد | الأولوية |
|---|--------|---------|--------|----------|

## 3. أبرز النقاط المطروحة
- نقطة 1
- نقطة 2
- نقطة 3

## 4. خريطة ذهنية
- **الموضوع الرئيسي**: ...
  - **فرع 1**: ...
    - تفصيل 1
    - تفصيل 2
  - **فرع 2**: ...
    - تفصيل 1
  - **فرع 3**: ...
    - تفصيل 1

## 5. الخلاصة التنفيذية
(5-7 أسطر موجزة تصف محور الاجتماع ونتائجه)

## 6. المصطلحات الإنجليزية
(اذكر المصطلحات الإنجليزية الواردة مع شرحها، أو اكتب: لا يوجد)

ابدأ مباشرة بالعنوان الأول دون أي مقدمة.
"""
    t0 = time.time()
    resp = qwen_client.chat.completions.create(
        model=QWEN_MODEL_ID,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=4000,
        temperature=0.4,
        extra_body={"options": {"num_ctx": 16384}}
    )
    elapsed = time.time() - t0
    result  = resp.choices[0].message.content.strip()
    print(f"   ✅ Qwen3 أنتج {len(result)} حرف في {elapsed:.0f}ث")
    return result


def hybrid_summary(full_text):
    print("🔍 Fanar: جاري قراءة وتحليل النص العربي كاملاً ...")
    try:
        fanar_analysis = fanar_understand(full_text)
    except Exception as e:
        print(f"   ⚠️  Fanar فشل: {e}")
        fanar_analysis = ""
    print("🧠 Qwen3: جاري بناء التقرير الهيكلي ...")
    return qwen_report(fanar_analysis, full_text)


print("\n🤖 بدء التلخيص الهجين المحلي: Fanar → Qwen3 ...")
summary_text = hybrid_summary(full_transcript)

SUMMARY_MD  = f"{OUTPUT_DIR}/{SESSION}_summary.md"
SUMMARY_TXT = f"{OUTPUT_DIR}/{SESSION}_summary.txt"
with open(SUMMARY_MD,  "w", encoding="utf-8") as f: f.write(summary_text)
with open(SUMMARY_TXT, "w", encoding="utf-8") as f: f.write(summary_text)
print(f"\n💾 Markdown: {SUMMARY_MD}")
print(f"💾 TXT:      {SUMMARY_TXT}")


✅ Ollama متصل | النماذج المتاحة: ['fanar-8k:latest', 'qwen3:latest', 'hf.co/mradermacher/Fanar-1-9B-Instruct-GGUF:Q4_K_M', 'fanar:latest']
⚠️  fanar-8k غير موجود — شغّل: ollama pull fanar-8k
✅ Fanar جاهز  | fanar-8k
✅ Qwen3 جاهز  | qwen3:latest

🤖 بدء التلخيص الهجين المحلي: Fanar → Qwen3 ...
🔍 Fanar: جاري قراءة وتحليل النص العربي كاملاً ...
🧠 Qwen3: جاري بناء التقرير الهيكلي ...
   ✅ Qwen3 أنتج 2321 حرف في 270ث

💾 Markdown: ./meeting_output/20260718_140022_summary.md
💾 TXT:      ./meeting_output/20260718_140022_summary.txt


In [9]:
# ════════════════════════════════════════════════════
# CELL 7 — عرض التقرير مرئياً: جداول + خريطة هرمية
# يفتح في المتصفح تلقائياً عند تشغيل Kernel VS Code
# ════════════════════════════════════════════════════
import re, os, webbrowser

if 'summary_text' not in globals() or not summary_text.strip():
    print("❌ summary_text غير موجود — شغّل Cell 6 أولاً")
else:
    def extract_section(text, heading):
        pattern = r'##\s*\d*\.?\s*' + re.escape(heading) + r'.*?\n(.*?)(?=\n##|\Z)'
        m = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
        return m.group(1).strip() if m else ''

    def md_table_to_html(md_table, hdr_color='#1B3A6B'):
        lines = [l.strip() for l in md_table.strip().splitlines() if l.strip()]
        rows  = [l for l in lines if not re.match(r'^\|[-| ]+\|$', l)]
        if not rows: return "<p style='color:#888'>لا توجد بيانات</p>"
        html = '<table style="width:100%;border-collapse:collapse;direction:rtl;font-size:14px;margin:8px 0">'
        for ri, row in enumerate(rows):
            cells = [c.strip() for c in row.strip('|').split('|')]
            tag   = 'th' if ri == 0 else 'td'
            bg    = hdr_color if ri == 0 else ('#f0f7ff' if ri % 2 == 0 else '#ffffff')
            color = '#ffffff' if ri == 0 else '#1e293b'
            html += '<tr>' + ''.join(
                f'<{tag} style="padding:8px 12px;border:1px solid #cbd5e1;'
                f'background:{bg};color:{color};text-align:right">{c}</{tag}>'
                for c in cells) + '</tr>'
        return html + '</table>'

    def build_mind_map(md_text):
        lines = [l for l in md_text.splitlines() if l.strip()]
        if not lines: return "<p style='color:#888'>لا توجد بيانات</p>"
        main_topic, branches, current_branch = '', [], None
        for line in lines:
            s = line.strip()
            if 'الموضوع الرئيسي' in s:
                m = re.search(r'\*\*[^*]+\*\*[:\s]*(.*)', s)
                main_topic = m.group(1).strip().strip('*') if m else re.sub(r'[-*#]','',s).strip()
            elif re.match(r'^\s{0,6}-\s+\*\*فرع|^\s{2,4}-\s+\*\*', line) and 'الموضوع' not in s:
                m = re.search(r'\*\*([^*]+)\*\*[:\s]*(.*)', s)
                if m:
                    current_branch = {'name': m.group(1).strip(), 'detail': m.group(2).strip(), 'children': []}
                    branches.append(current_branch)
            elif re.match(r'^\s{6,}-', line) and current_branch is not None:
                child = re.sub(r'^\s+-\s*', '', s)
                child = re.sub(r'\*\*([^*]+)\*\*', r'\1', child)
                current_branch['children'].append(child)
        if not main_topic and lines:
            main_topic = re.sub(r'[-*#]', '', lines[0]).strip()
        colors = ['#2563EB','#059669','#B45309','#7C3AED','#DC2626','#0891B2']
        branches_html = ''
        for i, br in enumerate(branches):
            col     = colors[i % len(colors)]
            ch_html = ''.join(
                f'<div style="background:#f8fafc;border:1px solid {col}40;border-right:3px solid {col};'
                f'border-radius:6px;padding:5px 10px;margin:4px 0;font-size:12px;color:#334155;text-align:right">{c}</div>'
                for c in br['children'])
            detail  = f"<br><span style='font-size:11px;opacity:.9'>{br['detail']}</span>" if br['detail'] else ''
            vline   = f"<div style='width:2px;height:16px;background:{col}40'></div>" if ch_html else ''
            branches_html += (
                f'<div style="display:flex;flex-direction:column;align-items:center;min-width:150px;max-width:190px">'
                f'<div style="background:{col};color:white;border-radius:10px;padding:10px 14px;'
                f'font-size:13px;font-weight:bold;text-align:center;width:100%;box-sizing:border-box;'
                f'box-shadow:0 2px 8px {col}40">{br["name"]}{detail}</div>'
                f'{vline}<div style="width:100%">{ch_html}</div></div>')
        cw = '80%' if len(branches) > 1 else '0'
        return (
            '<div style="direction:rtl;text-align:center;padding:16px;'
            'background:linear-gradient(135deg,#f0f9ff,#e0f2fe);'
            'border-radius:16px;border:2px solid #0ea5e9;margin:12px 0">'
            '<div style="font-weight:bold;font-size:18px;color:#1B3A6B;background:white;'
            'border:3px solid #1B3A6B;border-radius:12px;padding:12px 24px;display:inline-block;'
            f'box-shadow:0 4px 12px rgba(27,58,107,.15);margin-bottom:16px">{main_topic}</div>'
            '<div style="width:2px;height:20px;background:#1B3A6B;margin:0 auto"></div>'
            f'<div style="width:{cw};height:2px;background:#94a3b8;margin:0 auto"></div>'
            f'<div style="display:flex;justify-content:center;gap:12px;flex-wrap:wrap;margin-top:0">{branches_html}</div>'
            '</div>')

    def bullets_to_html(text):
        items = [re.sub(r'^-\s*','',l.strip()) for l in text.splitlines() if l.strip().startswith('-')]
        if not items: return f'<p>{text}</p>'
        return "<ul style='text-align:right;padding-right:20px;color:#1e293b'>" +                ''.join(f"<li style='margin:6px 0;font-size:14px'>{i}</li>" for i in items) + '</ul>'

    sec_decisions = extract_section(summary_text, 'جدول القرارات الرئيسية')
    sec_tasks     = extract_section(summary_text, 'جدول المهام والمسؤوليات')
    sec_points    = extract_section(summary_text, 'أبرز النقاط المطروحة')
    sec_mindmap   = extract_section(summary_text, 'خريطة ذهنية')
    sec_executive = extract_section(summary_text, 'الخلاصة التنفيذية')
    sec_english   = extract_section(summary_text, 'المصطلحات الإنجليزية')

    eng_block = ''
    if sec_english and 'لا يوجد' not in sec_english:
        eng_block = (
            '<h3 style="color:#1B3A6B;border-right:4px solid #0891B2;padding-right:10px;margin-top:20px">'
            '🌐 المصطلحات الإنجليزية</h3>'
            '<div style="background:#f0f9ff;border:1px solid #bae6fd;border-radius:10px;'
            'padding:14px 18px;font-size:14px;line-height:1.9;color:#1e293b">'
            + sec_english.replace('\n','<br>') + '</div>')

    html_content = f"""<!DOCTYPE html>
<html lang="ar" dir="rtl">
<head>
<meta charset="UTF-8">
<title>ملخص الاجتماع — {SESSION}</title>
<style>
  body{{font-family:'Segoe UI',Tahoma,Arial,sans-serif;background:#f1f5f9;
        direction:rtl;text-align:right;margin:0;padding:20px;color:#1e293b}}
  .container{{max-width:960px;margin:0 auto;background:white;border-radius:18px;
              padding:32px;box-shadow:0 4px 20px rgba(0,0,0,.08)}}
  h1{{color:#1a6e35;border-bottom:3px solid #28a745;padding-bottom:12px}}
  h3{{color:#1B3A6B}}
  .meta{{color:#666;font-size:13px;margin-top:-8px;margin-bottom:20px}}
  .exec-box{{background:#fff9f0;border:1px solid #fed7aa;border-radius:10px;
             padding:14px 18px;font-size:14.5px;line-height:1.9}}
  .badge{{display:inline-block;padding:4px 10px;border-radius:20px;font-size:12px;
          font-weight:bold;margin-left:8px}}
</style>
</head>
<body>
<div class="container">
  <h1>📋 الملخص النهائي</h1>
  <p class="meta">
    Session: {SESSION}
    <span class="badge" style="background:#e0f2fe;color:#0369a1">Fanar</span>
    <span style="font-size:18px">+</span>
    <span class="badge" style="background:#f0fdf4;color:#15803d">Qwen3</span>
    <span class="badge" style="background:#fef3c7;color:#92400e">محلي 100% 🖥️</span>
  </p>
  <hr style="border-color:#c3e6cb;margin-bottom:20px">

  <h3 style="border-right:4px solid #2563EB;padding-right:10px">📌 جدول القرارات الرئيسية</h3>
  {md_table_to_html(sec_decisions,'#1B3A6B')}

  <h3 style="border-right:4px solid #059669;padding-right:10px;margin-top:20px">✅ جدول المهام والمسؤوليات</h3>
  {md_table_to_html(sec_tasks,'#059669')}

  <h3 style="border-right:4px solid #B45309;padding-right:10px;margin-top:20px">💡 أبرز النقاط المطروحة</h3>
  {bullets_to_html(sec_points)}

  <h3 style="border-right:4px solid #7C3AED;padding-right:10px;margin-top:20px">🗺️ الخريطة الهرمية (Mind Map)</h3>
  {build_mind_map(sec_mindmap)}

  <h3 style="border-right:4px solid #DC2626;padding-right:10px;margin-top:20px">📝 الخلاصة التنفيذية</h3>
  <div class="exec-box">{sec_executive.replace(chr(10),'<br>')}</div>

  {eng_block}
</div>
</body></html>"""

    HTML_FILE = f"{OUTPUT_DIR}/{SESSION}_report.html"
    with open(HTML_FILE, "w", encoding="utf-8") as f:
        f.write(html_content)
    print(f"✅ HTML محفوظ: {HTML_FILE}")

    # فتح في المتصفح تلقائياً
    webbrowser.open(f"file://{os.path.abspath(HTML_FILE)}")
    print("🌐 تم فتح التقرير في المتصفح")
    print("\n─── معاينة نصية ───")
    print(summary_text[:1500] + ("..." if len(summary_text) > 1500 else ""))


✅ HTML محفوظ: ./meeting_output/20260718_140022_report.html
🌐 تم فتح التقرير في المتصفح

─── معاينة نصية ───
## 1. جدول القرارات الرئيسية  
| # | القرار | التفاصيل | المسؤول | الموعد |  
|---|--------|----------|---------|--------|  
| 1 | تحديد ميزانية البحث | الميزانية تتراوح بين 1000 إلى 3000 جنيه | الطالب | قبل بدء البحث |  
| 2 | إعداد بحث مسبق | لضمان جودة البحث وتجنب الأخطاء | المشرف الأكاديمي | قبل بدء البحث |  
| 3 | أهمية إنجاز البحث | البحث يجب أن يعكس تطبيق المعرفة وآلية التعلم | الطالب | خلال فترة الدراسة |  

---

## 2. جدول المهام والمسؤوليات  
| # | المهمة | المسؤول | الموعد | الأولوية |  
|---|--------|---------|--------|----------|  
| 1 | إعداد بحث مسبق | الطالب | قبل 3 أشهر | عالى |  
| 2 | تنفيذ البحث | الطالب | خلال فترة الدراسة | عالى |  
| 3 | كتابة البحث وتحليل النتائج | الطالب | قبل التسليم | عالى |  
| 4 | نشر البحث في مجلة | المشرف الأكاديمي | بعد الانتهاء | متوسط |  
| 5 | مراجعة البحث من قبل الجهة المشرفة | المشرف الأكاديمي | قبل النشر | عالى |  

---

## 3

In [10]:
# ════════════════════════════════════════════════════
# CELL 8 — تحرير الذاكرة
# ════════════════════════════════════════════════════
import gc, torch

for var in ['asr_model', 'audio_array']:
    if var in globals():
        try: del globals()[var]
        except: pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

show_ram()
print("✅ الذاكرة محررة — Pipeline اكتمل 🎉")


RAM  [██████████░░░░░░░░░░] 18.6/34.0 GB  (55%)
GPU  not available — CPU mode
✅ الذاكرة محررة — Pipeline اكتمل 🎉


## 📊 تقييم جودة الملخص (Summary Evaluation)

يقيّم هذا القسم جودة `summary_text` الناتج من Cell 6 باستخدام ثلاث مقاييس مكمّلة:

| المقياس | النوع | يقيس ماذا |
|---|---|---|
| **ROUGE-1/2/L** | تطابق لغوي (Lexical) | نسبة تطابق الكلمات/التتابعات بين الملخص والمرجع |
| **BERTScore** | تشابه دلالي (Semantic) | تشابه المعنى عبر نماذج لغوية |
| **LLM-as-a-Judge** | تقييم شبيه بالبشر | يقيّم Qwen3 (محلي) الدقة والشمولية والتماسك والإيجاز وجودة اللغة |

> 💡 إن تركت `REFERENCE_SUMMARY` فارغاً في Cell 1، سيُستخدم النص الكامل كمرجع تقريبي.


In [11]:
# ════════════════════════════════════════════════════
# EVAL 1 — 📏 ROUGE (تطابق لغوي)
# ════════════════════════════════════════════════════
import re
from rouge_score import rouge_scorer

class ArabicTokenizer:
    _tashkeel = re.compile(r'[\u064B-\u0652\u0670\u0640]')
    def tokenize(self, text):
        text = self._tashkeel.sub('', text)
        return re.findall(r'\w+', text, flags=re.UNICODE)

rouge_reference = REFERENCE_SUMMARY if REFERENCE_SUMMARY else full_transcript
HAS_REFERENCE   = bool(REFERENCE_SUMMARY)

if not HAS_REFERENCE:
    print("⚠️  لا يوجد ملخص مرجعي — سيُستخدم النص الكامل كمرجع تقريبي")
    print("   (ضع ملخصاً بشرياً في REFERENCE_SUMMARY بـ Cell 1 للنتائج الدقيقة)")

scorer_r   = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'],
                                       use_stemmer=False, tokenizer=ArabicTokenizer())
rouge_scores = scorer_r.score(rouge_reference, summary_text)

label = "مقابل الملخص المرجعي" if HAS_REFERENCE else "مقابل النص الكامل (تقريبي)"
print(f"\n📏 ROUGE Scores — {label}")
print("─" * 60)
print(f"{'Metric':<10} | {'Precision':>10} | {'Recall':>10} | {'F1':>10}")
print("─" * 60)
for metric, s in rouge_scores.items():
    print(f"{metric.upper():<10} | {s.precision:>10.4f} | {s.recall:>10.4f} | {s.fmeasure:>10.4f}")


⚠️  لا يوجد ملخص مرجعي — سيُستخدم النص الكامل كمرجع تقريبي
   (ضع ملخصاً بشرياً في REFERENCE_SUMMARY بـ Cell 1 للنتائج الدقيقة)

📏 ROUGE Scores — مقابل النص الكامل (تقريبي)
────────────────────────────────────────────────────────────
Metric     |  Precision |     Recall |         F1
────────────────────────────────────────────────────────────
ROUGE1     |     0.4745 |     0.0052 |     0.0102
ROUGE2     |     0.0511 |     0.0006 |     0.0011
ROUGEL     |     0.2229 |     0.0024 |     0.0048


In [12]:
# ════════════════════════════════════════════════════
# EVAL 2 — 🧠 BERTScore (تشابه دلالي)
# ════════════════════════════════════════════════════
import torch
from bert_score import score as bertscore_fn

bertscore_reference = REFERENCE_SUMMARY if HAS_REFERENCE else full_transcript

if not HAS_REFERENCE:
    print("⚠️  BERTScore: مقارنة تقريبية بالنص الكامل (أول ~512 token)")

bs_device = "cuda" if torch.cuda.is_available() else "cpu"
P, R, F1  = bertscore_fn([summary_text], [bertscore_reference],
                          lang="ar", device=bs_device, verbose=True)

label = "مقابل الملخص المرجعي" if HAS_REFERENCE else "مقابل النص الكامل (تقريبي)"
print(f"\n🧠 BERTScore — {label}")
print("─" * 40)
print(f"  Precision : {P.item():.4f}")
print(f"  Recall    : {R.item():.4f}")
print(f"  F1        : {F1.item():.4f}")


⚠️  BERTScore: مقارنة تقريبية بالنص الكامل (أول ~512 token)


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cpu).
W0718 16:27:15.421000 16608 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 1.20 seconds, 0.83 sentences/sec

🧠 BERTScore — مقابل النص الكامل (تقريبي)
────────────────────────────────────────
  Precision : 0.5921
  Recall    : 0.5797
  F1        : 0.5858


In [13]:
# ════════════════════════════════════════════════════
# EVAL 3 — ⚖️ LLM-as-a-Judge (Qwen3 محلي كحكم مستقل)
# Qwen3 لم يشارك في الفهم (فعل ذلك Fanar) — حكم أكثر استقلالية
# ════════════════════════════════════════════════════
import json, re, time
from openai import OpenAI

judge_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")
print(f"✅ LLM Judge جاهز | {QWEN_MODEL_ID} (محلي — حكم مستقل)")

ref_block = (f"\n**ملخص مرجعي (للمقارنة):**\n{REFERENCE_SUMMARY}\n"
             if REFERENCE_SUMMARY else "")

judge_prompt = f"""أنت مقيّم خبير ومحايد لملخصات اجتماعات الأعمال باللغة العربية.

**النص الكامل للاجتماع (الحقيقة المرجعية):**
{full_transcript[:12000]}
{ref_block}
**الملخص المطلوب تقييمه:**
{summary_text}

قيّم الملخص أعلاه على المعايير التالية، كل معيار من 1 إلى 10:
1. faithfulness (الدقة والأمانة): هل المعلومات صحيحة وغير مُختلقة؟
2. coverage (الشمولية): هل يغطي أهم القرارات والمهام والنقاط؟
3. coherence (التماسك والوضوح): هل التنظيم والصياغة مفهومة ومنطقية؟
4. conciseness (الإيجاز): هل الملخص موجز دون حشو أو تكرار؟
5. language_quality (جودة اللغة العربية): هل اللغة سليمة نحويًا وأسلوبيًا؟

أعد إجابتك بصيغة JSON فقط، بدون أي شرح أو نص خارج JSON:
{{
  "faithfulness":     {{"score": 0, "justification": "..."}},
  "coverage":         {{"score": 0, "justification": "..."}},
  "coherence":        {{"score": 0, "justification": "..."}},
  "conciseness":      {{"score": 0, "justification": "..."}},
  "language_quality": {{"score": 0, "justification": "..."}},
  "overall_score": 0,
  "overall_comment": "..."
}}"""

judge_result = {}
print("⚖️  جاري التقييم ...")
for attempt in range(5):
    try:
        resp = judge_client.chat.completions.create(
            model=QWEN_MODEL_ID,
            messages=[{"role": "user", "content": judge_prompt}],
            temperature=0.1, max_tokens=1500)
        raw = resp.choices[0].message.content.strip()
        raw = re.sub(r"^```json\s*|\s*```$", "", raw, flags=re.MULTILINE).strip()
        judge_result = json.loads(raw)
        break
    except json.JSONDecodeError:
        print(f"   ⚠️  JSON غير صالح، محاولة {attempt+1}/5 ...")
        time.sleep(5)
    except Exception as e:
        print(f"   ❌ خطأ: {e}")
        break

criteria_ar = {
    "faithfulness":     "الدقة والأمانة",
    "coverage":         "الشمولية",
    "coherence":        "التماسك والوضوح",
    "conciseness":      "الإيجاز",
    "language_quality": "جودة اللغة",
}
print("\n📊 نتائج LLM Judge")
print("─" * 60)
for key, label_ar in criteria_ar.items():
    item = judge_result.get(key, {})
    print(f"{label_ar:<18} | {item.get('score','-')}/10")
    print(f"   └─ {item.get('justification','')}")
print("─" * 60)
print(f"{'التقييم العام':<18} | {judge_result.get('overall_score','-')}/10")
print(f"   └─ {judge_result.get('overall_comment','')}")


✅ LLM Judge جاهز | qwen3:latest (محلي — حكم مستقل)
⚖️  جاري التقييم ...

📊 نتائج LLM Judge
────────────────────────────────────────────────────────────
الدقة والأمانة     | 5/10
   └─ الملخص يحتوي على معلومات غير موجودة في النص الأصلي مثل تكلفة البحث (1000-3000 جنيه) وفقرات حول مراجعة الزملاء (Peer Review) والمجلات الدولية (International Journal) التي لم تُذكر في الحوار الأصلي.
الشمولية           | 8/10
   └─ يغطي النقاط الرئيسية مثل تكاليف البحث، أهمية البحث المسبق، وربط البحث بالتطبيق الأكاديمي، لكنه يفتقر إلى تفاصيل مثل التحديات العملية أو المعايير المحددة للبحث المسبق.
التماسك والوضوح    | 9/10
   └─ التنظيم منطقي مع فئات واضحة (جدول القرارات، المهام، النقاط المطروحة)، وربط الأفكار بشكل منهجي دون تضارب.
الإيجاز            | 7/10
   └─ يحتوي على تفاصيل مفيدة لكنه يكرر بعض المعلومات (مثل تكلفة البحث في أكثر من مكان) ويحتاج إلى تبسيط بعض الفقرات.
جودة اللغة         | 7/10
   └─ اللغة العربية سليمة نحويًا وأسلوبيًا، لكن هناك بعض التكرار في التعبيرات وفقرات غير مُفصَّلة بشكل كامل.
─────

In [ ]:
# ════════════════════════════════════════════════════
# EVAL 4 — 📋 تقرير التقييم النهائي + حفظ JSON
# ════════════════════════════════════════════════════
import json, os
from IPython.display import display, HTML

evaluation_report = {
    "session":               SESSION,
    "fanar_model":           FANAR_MODEL_ID,
    "qwen_model":            QWEN_MODEL_ID,
    "has_reference_summary": HAS_REFERENCE,
    "rouge": {
        m: {"precision": s.precision, "recall": s.recall, "f1": s.fmeasure}
        for m, s in rouge_scores.items()
    },
    "bertscore": {
        "precision": float(P.item()),
        "recall":    float(R.item()),
        "f1":        float(F1.item()),
    },
    "llm_judge": judge_result,
}

EVAL_JSON = f"{OUTPUT_DIR}/{SESSION}_evaluation.json"
with open(EVAL_JSON, "w", encoding="utf-8") as f:
    json.dump(evaluation_report, f, ensure_ascii=False, indent=2)
print(f"💾 تقرير التقييم محفوظ: {EVAL_JSON}\n")

ref_note = ("✅ مقارنة بملخص مرجعي بشري" if HAS_REFERENCE
            else "⚠️ ROUGE/BERTScore مقارنة تقريبية بالنص الكامل (لا يوجد مرجع بشري)")

display(HTML(f"""
<div dir="rtl" style="font-family:'Segoe UI',Tahoma,Arial,sans-serif;
    direction:rtl;text-align:right;padding:24px 28px;
    background:linear-gradient(135deg,#f8f9fa,#eef3ff);
    border-radius:18px;border:2px solid #4a6cf7;margin:12px 0">
  <h2 style="color:#2a3eb1;margin-bottom:14px">📊 تقرير تقييم الملخص</h2>
  <p style="font-size:13px;color:#555">
    <b>Fanar:</b> {FANAR_MODEL_ID} &nbsp;|&nbsp; <b>Qwen3:</b> {QWEN_MODEL_ID} &nbsp;|&nbsp; 🖥️ محلي 100%
  </p>
  <table style="width:100%;border-collapse:collapse;font-size:14.5px">
    <tr style="background:#dde6ff">
      <th style="padding:8px;text-align:right;border:1px solid #c5d1ff">المقياس</th>
      <th style="padding:8px;text-align:center;border:1px solid #c5d1ff">Precision</th>
      <th style="padding:8px;text-align:center;border:1px solid #c5d1ff">Recall</th>
      <th style="padding:8px;text-align:center;border:1px solid #c5d1ff">F1</th>
    </tr>
    <tr>
      <td style="padding:8px;border:1px solid #c5d1ff">ROUGE-1</td>
      <td style="padding:8px;text-align:center;border:1px solid #c5d1ff">{rouge_scores['rouge1'].precision:.3f}</td>
      <td style="padding:8px;text-align:center;border:1px solid #c5d1ff">{rouge_scores['rouge1'].recall:.3f}</td>
      <td style="padding:8px;text-align:center;border:1px solid #c5d1ff">{rouge_scores['rouge1'].fmeasure:.3f}</td>
    </tr>
    <tr style="background:#f0f4ff">
      <td style="padding:8px;border:1px solid #c5d1ff">ROUGE-2</td>
      <td style="padding:8px;text-align:center;border:1px solid #c5d1ff">{rouge_scores['rouge2'].precision:.3f}</td>
      <td style="padding:8px;text-align:center;border:1px solid #c5d1ff">{rouge_scores['rouge2'].recall:.3f}</td>
      <td style="padding:8px;text-align:center;border:1px solid #c5d1ff">{rouge_scores['rouge2'].fmeasure:.3f}</td>
    </tr>
    <tr>
      <td style="padding:8px;border:1px solid #c5d1ff">ROUGE-L</td>
      <td style="padding:8px;text-align:center;border:1px solid #c5d1ff">{rouge_scores['rougeL'].precision:.3f}</td>
      <td style="padding:8px;text-align:center;border:1px solid #c5d1ff">{rouge_scores['rougeL'].recall:.3f}</td>
      <td style="padding:8px;text-align:center;border:1px solid #c5d1ff">{rouge_scores['rougeL'].fmeasure:.3f}</td>
    </tr>
    <tr style="background:#f0f4ff">
      <td style="padding:8px;border:1px solid #c5d1ff">BERTScore</td>
      <td style="padding:8px;text-align:center;border:1px solid #c5d1ff">{P.item():.3f}</td>
      <td style="padding:8px;text-align:center;border:1px solid #c5d1ff">{R.item():.3f}</td>
      <td style="padding:8px;text-align:center;border:1px solid #c5d1ff">{F1.item():.3f}</td>
    </tr>
    <tr style="background:#e8ecff">
      <td style="padding:8px;border:1px solid #c5d1ff"><b>LLM Judge (عام)</b></td>
      <td colspan="3" style="padding:8px;text-align:center;border:1px solid #c5d1ff">
        <b>{judge_result.get('overall_score','-')}/10</b>
        — {judge_result.get('overall_comment','')}
      </td>
    </tr>
  </table>
  <p style="color:#666;font-size:12.5px;margin-top:12px">{ref_note}</p>
</div>
"""))

print(f"\n{'='*55}")
print(f"  ✅ التقييم اكتمل!")
print(f"{'='*55}")
print(f"  📁 كل الملفات في: {os.path.abspath(OUTPUT_DIR)}/")
print(f"  📜 {SESSION}_transcript.txt")
print(f"  📋 {SESSION}_summary.md")
print(f"  🌐 {SESSION}_report.html")
print(f"  📊 {SESSION}_evaluation.json")
print(f"{'='*55}")
